# 🎫 AI Support Triage Agent — Agentic Tool-Calling System

## A GenAI / Agentic Systems Project

**Author:** Preeti Bhardwaj
**Domain:** Customer Support / SaaS Operations
**Tools:** Python, LangChain, FAISS, Groq LLaMA 3.3-70B (native tool-calling), Hugging Face MiniLM, Streamlit

---

## 🎯 Objective
Most "GenAI support bot" demos are a single LLM call answering a question. That breaks down fast in production: the model has no way to say "I don't know, route this to a human," no way to ask for missing information, and no structured trail an ops team can audit.

This project builds a **tool-calling agent** that reasons through a support ticket in multiple steps — classify, retrieve, decide, act — and explicitly chooses between resolving, escalating, or asking a clarifying question, instead of always generating *an* answer regardless of confidence.

## 📋 Project Workflow
1. Knowledge Base Setup (FAQ + policy docs → FAISS index)
2. Tool Schema Definition (the agent's available actions)
3. Agent Loop — Native Tool-Calling
4. Escalation & Confidence Logic
5. Synthetic Ticket Dataset
6. Evaluation Harness (routing accuracy + escalation precision)
7. Key Findings & Design Decisions

---


## 📦 Section 1: Setup & Installation

In [ ]:
# Install required libraries
!pip install langchain langchain-groq faiss-cpu sentence-transformers groq -q
print("All libraries installed ✅")

In [ ]:
# Import all libraries
import json
import os
import re
from datetime import datetime
from getpass import getpass

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq

print("All libraries imported ✅")

In [ ]:
# Set Groq API key
# Get a free key at https://console.groq.com
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq client ready ✅")

## 📚 Section 2: Knowledge Base — FAQ & Policy Documents

We simulate a small SaaS company's internal knowledge base: FAQ entries and policy snippets that
the agent retrieves from before deciding how to act. This reuses the same FAISS + embedding
pattern from the Medical RAG Assistant project, applied to a support-ops context.

**Why a small, hand-built KB instead of a huge scraped one?**
Keeping it small and fully visible lets us reason precisely about retrieval quality and write
an eval set where we *know* the ground truth — essential for the evaluation harness in Section 6.

In [ ]:
# Knowledge base: (doc_id, category, content)
knowledge_base = [
    ("KB001", "billing", "Refunds are issued for cancellations made within 14 days of purchase. "
        "Refunds are processed to the original payment method within 5-7 business days. "
        "Refunds outside the 14-day window require manager approval."),
    ("KB002", "billing", "To update a payment method, go to Settings > Billing > Payment Methods. "
        "Failed payments retry automatically 3 times over 7 days before the subscription is paused."),
    ("KB003", "billing", "Annual plans can be downgraded to monthly only at the renewal date. "
        "Mid-cycle downgrades are not supported and will not be prorated."),
    ("KB004", "account", "Password resets are self-service via the 'Forgot Password' link on the login page. "
        "Reset links expire after 30 minutes."),
    ("KB005", "account", "Two-factor authentication (2FA) can be enabled in Settings > Security. "
        "If a user is locked out due to lost 2FA access, this requires identity verification "
        "by the Security team and CANNOT be resolved by self-service or by the support bot."),
    ("KB006", "account", "Account deletion is permanent after a 30-day grace period. "
        "Users can cancel a deletion request within that window from Settings > Account."),
    ("KB007", "technical", "The mobile app requires iOS 15+ or Android 10+. "
        "If the app crashes on launch, clearing cache or reinstalling resolves 90% of cases."),
    ("KB008", "technical", "API rate limits are 100 requests/minute on the Starter plan and "
        "1000 requests/minute on the Pro plan. Rate limit errors return HTTP 429."),
    ("KB009", "technical", "CSV export is limited to 50,000 rows per file on all plans. "
        "Larger exports should be split by date range."),
    ("KB010", "feature", "Team workspaces support up to 10 members on the Pro plan and "
        "unlimited members on the Enterprise plan."),
    ("KB011", "feature", "Dark mode is available in Settings > Appearance on web and mobile."),
    ("KB012", "feature", "Custom integrations via webhook are available on Pro and Enterprise plans only."),
    ("KB013", "policy", "Any request involving a legal threat, mention of a lawyer, regulatory complaint, "
        "or data breach concern must be escalated to the Legal/Trust & Safety team immediately, "
        "regardless of how minor the underlying issue seems."),
    ("KB014", "policy", "Requests to delete a user's data under GDPR/CCPA must be escalated to the "
        "Data Privacy team. Support cannot process these directly even if technically possible."),
]

print(f"Knowledge base loaded: {len(knowledge_base)} documents ✅")
print(f"Categories: {sorted(set(d[1] for d in knowledge_base))}")

In [ ]:
# Embed the knowledge base with MiniLM (same model used in Medical RAG Assistant)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

kb_texts = [doc[2] for doc in knowledge_base]
kb_embeddings = embedder.encode(kb_texts, show_progress_bar=True)

# Build FAISS index
dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(kb_embeddings).astype("float32"))

print(f"✅ FAISS index built: {index.ntotal} vectors, dimension {dimension}")

In [ ]:
def search_knowledge_base(query: str, k: int = 3):
    """
    Retrieve the top-k most relevant KB documents for a query.
    Returns list of (doc_id, category, content, distance).
    Lower distance = more relevant (L2 distance on embeddings).
    """
    query_embedding = embedder.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        doc_id, category, content = knowledge_base[idx]
        results.append({
            "doc_id": doc_id,
            "category": category,
            "content": content,
            "distance": float(dist)
        })
    return results

# Quick sanity check
test_results = search_knowledge_base("how do I get my money back")
print("Test retrieval for 'how do I get my money back':\n")
for r in test_results:
    print(f"  [{r['doc_id']}] (dist={r['distance']:.3f}) {r['content'][:80]}...")

## 🛠️ Section 3: Tool Schema — The Agent's Available Actions

This is the core design decision of the whole project. Instead of one LLM call producing a reply,
the model is given a **fixed set of callable tools** and must explicitly choose one (or more) based
on its reasoning. This is what makes the system *agentic* rather than a single prompted response —
the same distinction between a RAG chatbot and an agent that decides what to do.

**The four tools, and the rule for each:**

| Tool | Purpose | Calls when... |
|---|---|---|
| `search_knowledge_base` | Retrieve relevant KB docs | Almost always called first to gather grounding |
| `draft_response` | Produce a customer-facing reply | KB coverage is direct, on-topic, and unambiguous |
| `escalate` | Route to a human team | KB coverage is weak, OR the ticket matches a hard policy trigger (security lockout, legal/GDPR), OR confidence is low |
| `ask_clarifying_question` | Request more info from the customer | The ticket is too vague to classify confidently |

**The hard escalation rule** mirrors the refusal-logic design from the Medical RAG Assistant:
*if the agent isn't confident, it must not guess — it escalates instead.* Two categories are
**always** escalated regardless of model confidence: account-security lockouts (KB005) and
legal/privacy requests (KB013, KB014). These are hard-coded policy triggers, not left to model
judgment, because the cost of a wrong guess there is too high to leave to a probabilistic call.

In [ ]:
# Tool schema in Groq's native function-calling format (OpenAI-compatible)
tools = [
    {
        "type": "function",
        "function": {
            "name": "draft_response",
            "description": "Draft a customer-facing reply that directly resolves the ticket using "
                            "knowledge base content. Only call this when the retrieved KB content "
                            "clearly and directly answers the customer's question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "response_text": {
                        "type": "string",
                        "description": "The customer-facing reply, written clearly and politely."
                    },
                    "source_doc_ids": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "KB doc IDs used to ground this response, e.g. ['KB001']."
                    },
                    "confidence": {
                        "type": "string",
                        "enum": ["high", "medium"],
                        "description": "How confident the agent is this fully resolves the ticket."
                    }
                },
                "required": ["response_text", "source_doc_ids", "confidence"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate",
            "description": "Escalate the ticket to a human team instead of answering directly. "
                            "Use this when KB coverage is weak or absent, when confidence is low, "
                            "or when the ticket involves account security lockouts, legal threats, "
                            "GDPR/data deletion requests, or anything with high risk if answered wrong.",
            "parameters": {
                "type": "object",
                "properties": {
                    "team": {
                        "type": "string",
                        "enum": ["billing", "technical", "security", "legal_privacy", "general"],
                        "description": "Which team should handle this."
                    },
                    "reason": {
                        "type": "string",
                        "description": "Brief explanation of why this needs human handling."
                    },
                    "priority": {
                        "type": "string",
                        "enum": ["low", "medium", "high"],
                        "description": "Urgency of the escalation."
                    }
                },
                "required": ["team", "reason", "priority"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "ask_clarifying_question",
            "description": "Ask the customer a clarifying question instead of guessing, when the "
                            "ticket is too vague or ambiguous to classify or resolve confidently.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "The clarifying question to send back to the customer."
                    }
                },
                "required": ["question"]
            }
        }
    }
]

print(f"✅ {len(tools)} tools defined: {[t['function']['name'] for t in tools]}")

## 🤖 Section 4: The Agent Loop

The agent loop is intentionally a **fixed two-step pipeline**, not an open-ended "let the model
decide everything" loop:

1. **Always retrieve first.** `search_knowledge_base` is called directly by code (not left to the
   model to decide whether to search) — this guarantees the model never reasons about a ticket
   without grounding, the same way the Medical RAG Assistant always retrieves before generating.
2. **Then let the model choose an action.** The retrieved context, the hard policy rules, and the
   ticket are passed to the model with the three action tools. The model must call exactly one.

This hybrid design (deterministic retrieval + agentic decision) is a deliberate choice: full
freedom for the model to decide *whether* to search adds risk for no real benefit here, while
freedom to decide *what to do with* the results is where the real reasoning value is.

In [ ]:
SYSTEM_PROMPT = """You are a support triage agent for a SaaS company. Your job is to read an \
incoming support ticket along with retrieved knowledge base context, then choose EXACTLY ONE \
tool call: draft_response, escalate, or ask_clarifying_question.

Hard rules (always follow, regardless of how confident you feel):
- If the ticket mentions being locked out due to lost 2FA/two-factor access, you MUST escalate \
  to the security team. Never attempt to resolve this yourself.
- If the ticket mentions a lawyer, legal action, regulatory complaint, data breach, or a request \
  to delete personal data under GDPR/CCPA, you MUST escalate to legal_privacy team.
- If the retrieved knowledge base context does not clearly and directly answer the ticket, \
  escalate rather than guessing. A partial or tangential match is not sufficient.
- If the ticket is too vague to know what the customer actually needs, ask a clarifying question \
  instead of guessing.
- Only use draft_response when you are actually confident the KB content resolves the ticket.

Always cite which KB doc_ids you used if you draft a response.
"""

def run_agent(ticket_text: str, verbose: bool = True):
    """
    Run the full triage pipeline on a single ticket:
    1. Retrieve KB context
    2. Let the model choose a tool call based on ticket + context + hard rules
    3. Parse and return a structured result
    """
    # Step 1: deterministic retrieval (always happens, not left to model judgment)
    retrieved = search_knowledge_base(ticket_text, k=3)
    context_block = "\n".join(
        f"[{r['doc_id']}] (category: {r['category']}) {r['content']}" for r in retrieved
    )

    user_message = f"""Support ticket:
\"\"\"{ticket_text}\"\"\"

Retrieved knowledge base context:
{context_block}

Choose the single correct tool call for this ticket."""

    # Step 2: agent decides which tool to call
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ],
        tools=tools,
        tool_choice="required",
        temperature=0.1
    )

    message = response.choices[0].message

    if not message.tool_calls:
        # Should not happen with tool_choice="required", but handle gracefully
        return {"action": "error", "raw": message.content, "retrieved_docs": [r["doc_id"] for r in retrieved]}

    tool_call = message.tool_calls[0]
    action = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    result = {
        "ticket": ticket_text,
        "action": action,
        "args": args,
        "retrieved_docs": [r["doc_id"] for r in retrieved],
        "retrieval_top_distance": retrieved[0]["distance"] if retrieved else None
    }

    if verbose:
        print(f"📥 Ticket: {ticket_text[:70]}...")
        print(f"   Retrieved: {result['retrieved_docs']}")
        print(f"   ➡️  Action: {action}")
        print(f"   Args: {json.dumps(args, indent=2)}")
        print()

    return result

print("✅ Agent loop defined")

In [ ]:
# Quick smoke test on a few hand-written tickets covering each expected action
sample_tickets = [
    "I want a refund for my subscription, I cancelled it 3 days ago.",
    "I lost access to my authenticator app and can't log in to my account anymore.",
    "My lawyer says I have grounds to sue you for how you handled my data.",
    "the app is weird",
]

for t in sample_tickets:
    run_agent(t)

## 🧪 Section 5: Synthetic Ticket Dataset for Evaluation

A demo that "looks like it works" on a handful of cherry-picked examples isn't an evaluation.
This section builds a labeled dataset of **support tickets with known-correct actions**, so the
agent's behavior can be measured rather than eyeballed — the same principle behind the labeled
5-fold cross-validation in the PCOS and Hospital Readmission projects, applied to an agentic
system instead of a classifier.

Each ticket is labeled with the **expected action** a careful human triager would take. This lets
us compute real metrics in Section 6: routing accuracy, and specifically **escalation recall** —
how reliably the agent escalates things it truly should never answer on its own.

In [ ]:
# Labeled eval set: (ticket_text, expected_action, expected_team_if_escalate)
eval_set = [
    # --- Should draft_response (clear KB coverage) ---
    ("How do I update my credit card on file?", "draft_response", None),
    ("Is there a dark mode option in the app?", "draft_response", None),
    ("What's the API rate limit on the Pro plan?", "draft_response", None),
    ("Can I export more than 50000 rows in one CSV?", "draft_response", None),
    ("How many people can I add to my team workspace on Pro?", "draft_response", None),
    ("I forgot my password, how do I reset it?", "draft_response", None),
    ("Can I downgrade my annual plan to monthly right now?", "draft_response", None),
    ("Does the mobile app work on Android 9?", "draft_response", None),

    # --- Should escalate: hard policy triggers (security / legal) ---
    ("I lost my phone and can't access my 2FA codes, I'm completely locked out.", "escalate", "security"),
    ("My authenticator app broke and now I can't get into my account at all.", "escalate", "security"),
    ("I'm going to contact my attorney about how you've been storing my information.", "escalate", "legal_privacy"),
    ("Please delete all my personal data, I'm invoking my GDPR rights.", "escalate", "legal_privacy"),
    ("This is a formal complaint, I'm filing with the regulator if this isn't fixed.", "escalate", "legal_privacy"),
    ("There's been a data breach on your end and I want to know what you're doing about it.", "escalate", "legal_privacy"),

    # --- Should escalate: outside KB coverage / low confidence ---
    ("Can you migrate my data from a competitor's tool automatically?", "escalate", "general"),
    ("I want a custom enterprise contract with different SLA terms.", "escalate", "general"),
    ("Your billing charged me twice this month and the amounts don't match my invoice.", "escalate", "billing"),

    # --- Should ask_clarifying_question: too vague ---
    ("It's not working.", "ask_clarifying_question", None),
    ("the app is weird", "ask_clarifying_question", None),
    ("help", "ask_clarifying_question", None),
    ("something is broken with my account", "ask_clarifying_question", None),
]

print(f"✅ Eval set built: {len(eval_set)} labeled tickets")
print(f"   draft_response: {sum(1 for t in eval_set if t[1]=='draft_response')}")
print(f"   escalate: {sum(1 for t in eval_set if t[1]=='escalate')}")
print(f"   ask_clarifying_question: {sum(1 for t in eval_set if t[1]=='ask_clarifying_question')}")

## 📊 Section 6: Evaluation Harness

Two metrics matter here, and they are not equally important:

- **Overall routing accuracy** — did the agent pick the right action across all ticket types?
- **Escalation recall on hard-trigger tickets** — of all tickets that *must* be escalated
  (security lockouts, legal/GDPR), what fraction actually were? This is the high-stakes metric.
  A wrong `draft_response` on a security lockout ticket is a much worse failure than a wrong
  `draft_response` on "what's the rate limit" — so this gets measured separately, the same way
  the PCOS project optimized for recall over raw accuracy because false negatives were the
  costly error type.

In [ ]:
def evaluate_agent(eval_set):
    results = []
    for ticket_text, expected_action, expected_team in eval_set:
        result = run_agent(ticket_text, verbose=False)
        actual_action = result["action"]
        actual_team = result["args"].get("team") if actual_action == "escalate" else None

        action_correct = (actual_action == expected_action)
        team_correct = (expected_team is None) or (actual_team == expected_team)

        results.append({
            "ticket": ticket_text,
            "expected_action": expected_action,
            "actual_action": actual_action,
            "expected_team": expected_team,
            "actual_team": actual_team,
            "action_correct": action_correct,
            "team_correct": team_correct if action_correct else None,
        })
    return pd.DataFrame(results)

print("Running evaluation across all labeled tickets...\n")
eval_results = evaluate_agent(eval_set)

overall_accuracy = eval_results["action_correct"].mean()
print(f"\n{'='*60}")
print(f"Overall routing accuracy: {overall_accuracy:.1%}")

# The high-stakes metric: hard-trigger escalation recall
hard_trigger_tickets = eval_results[eval_results["expected_team"].isin(["security", "legal_privacy"])]
hard_trigger_recall = hard_trigger_tickets["action_correct"].mean()
print(f"Hard-trigger escalation recall (security/legal): {hard_trigger_recall:.1%}")
print(f"{'='*60}")

display(eval_results)

In [ ]:
# Breakdown by expected action type
print("Accuracy by expected action type:\n")
breakdown = eval_results.groupby("expected_action")["action_correct"].agg(["mean", "count"])
breakdown.columns = ["accuracy", "n_tickets"]
display(breakdown)

# Show any misclassified tickets for error analysis
errors = eval_results[~eval_results["action_correct"]]
if len(errors) > 0:
    print(f"\n⚠️ {len(errors)} misclassified ticket(s):\n")
    display(errors[["ticket", "expected_action", "actual_action"]])
else:
    print("\n✅ No misclassifications — all tickets routed correctly.")

## ✅ Section 7: Key Findings & Design Decisions

### 🔬 What this demonstrates that a single-prompt RAG chatbot doesn't
1. **Real tool-calling, not prompt chaining** — the model explicitly selects from a fixed action
   set via native function-calling, rather than producing free text that gets parsed afterward.
2. **A genuine refusal/escalation mechanism applied to actions, not just answers** — directly
   extending the hallucination-refusal design from the Medical RAG Assistant into an agentic
   context where the "refusal" is a routing decision, not just a withheld answer.
3. **Deterministic guardrails layered on top of model judgment** — security and legal triggers
   are never left purely to model confidence; they're hard-coded rules the model is instructed
   to follow regardless of how it reads the ticket. This reflects a production mindset: don't
   trust probabilistic judgment alone for the highest-cost failure modes.
4. **A real evaluation harness**, not a handful of cherry-picked demo examples — with a metric
   (hard-trigger escalation recall) chosen specifically because it's the costliest failure mode,
   the same recall-over-accuracy framing used in the PCOS and Hospital Readmission projects.

### ⚠️ Limitations & honest next steps
- The knowledge base is small and hand-built; a production system would need much broader
  coverage and a process for keeping it current as policies change.
- The eval set is synthetic and hand-labeled by one person (me) — a production system would
  validate against real historical tickets and inter-rater agreement on labels.
- The agent currently makes one tool call per ticket. A more advanced version could support
  multi-step tool use (e.g., search again with a refined query before deciding).
- No conversation memory yet — each ticket is evaluated independently, not as part of an
  ongoing multi-turn conversation with a customer.

### 🔧 What I'd build next
- Multi-turn conversation handling, so `ask_clarifying_question` actually continues the loop
  once the customer replies, instead of ending the interaction.
- A confidence-calibration check: compare the model's self-reported `confidence` field against
  actual correctness, to see if "high confidence" claims are trustworthy or need a lower bar
  for escalation.
- Swap the hand-built KB for a larger, real FAQ corpus and re-run the eval harness to see how
  routing accuracy holds up at scale.
